In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
from fuzzywuzzy import fuzz
import warnings
import os
import pickle

from linearmodels.panel import model
from stargazer.stargazer import Stargazer


In [2]:
temp_df = pd.read_excel('правильное наименование.xlsx')
dict_key = {n:k for k, n in zip(temp_df['key'], temp_df['name'])}
infl = pd.read_excel('infl_fed.xls')
list_months = infl.iloc[2, 83:].to_list()
dict_months_to_num = {k:f'0{i}' if i // 10 == 0 else f'{i}' for k, i in zip(infl.iloc[2, 83:].unique().tolist(), range(1,13))}
list_col = ['region', 'type_of_infl']
for year in range(2017, 2026):
    temp_list = list_months[12*(year-2017):12*(year-2016)]
    list_col += [dict_months_to_num[i] + f"_{year}" for i in temp_list]
infl_df = infl.drop(infl.columns[2:83], axis=1).iloc[4:].reset_index(drop=True).set_axis(list_col, axis=1).copy(deep = True)
list_regions_infl = [i.lstrip(' ') for i in infl_df['region'].unique()]
infl_df = infl_df[(infl_df.loc[:, 'type_of_infl'] == 'Все товары и услуги')].copy(deep = True).drop(['type_of_infl', 'region'], axis=1)
infl_df = infl_df.loc[[i in list(dict_key.keys()) for i in list_regions_infl]].copy()
infl_df = infl_df.reset_index(drop=True).set_axis([dict_key[k] for k in list_regions_infl if k in list(dict_key.keys())]).copy(deep=True)
df = pd.read_pickle("bf.pkl")
df = df[~(df.index.get_level_values('region') == 'ханты-мансийский автономный округ')].copy(deep=True)
df = df[~(df.index.get_level_values('region') == 'ямало-ненецкий автономный округ')].copy(deep=True)
list_regions = np.unique(df.index.get_level_values(0)).tolist()
for year in range(2017, 2025):
    temp_df = infl_df.iloc[:, 12*(year-2017):12*(year-2016)].copy()/100
    infl_df[year] = (temp_df.prod(axis = 1) - 1)*100
infl_df = infl_df.iloc[:, -8:].copy(deep=True)
df['infl'] = 0.0
df['infl_lag'] = 0.0
df['infl_lag2'] = 0.0
for region in df.index.get_level_values('region'):
    df.loc[region, 'infl'] = infl_df.loc[region][2:-1].astype(float).round(4).to_numpy()
    df.loc[region, 'infl_lag'] = infl_df.loc[region][1:-2].astype(float).round(4).to_numpy()
    df.loc[region, 'infl_lag2'] = infl_df.loc[region][:-3].astype(float).round(4).to_numpy()

In [4]:
df.loc[:, 'infl']

region                      year
белгородская область        2019     2.7959
                            2020     4.9338
                            2021     9.1164
                            2022    12.7942
                            2023     7.0094
                                     ...   
чукотский автономный округ  2019     3.7961
                            2020     1.9332
                            2021     5.7766
                            2022     5.6656
                            2023     4.8364
Name: infl, Length: 415, dtype: float64

In [31]:
infl_df.loc[region][2:-1].astype(float).round(4)

2019     2.7959
2020     4.9338
2021     9.1164
2022    12.7942
2023     7.0094
Name: белгородская область, dtype: float64

In [18]:
infl_df.iloc[:, -17:]

,10_2025,11_2025,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
белгородская область,100.42,100.63,9.026827,5.527599,6.244222,6.265667,16.211763,5.297077,2.967018,1.71667,5.623762,2.095799,6.024438,15.994729,5.326871,7.287821,9.794796
брянская область,100.23,100.33,10.614606,5.843349,7.536419,7.137855,18.292213,6.815012,5.362246,2.329661,6.217615,2.90344,6.465433,15.302694,7.10469,7.41065,10.891151
владимирская область,100.55,100.36,9.793131,5.75689,6.754965,7.49085,19.30489,5.504527,3.532314,2.17518,6.098148,2.49868,5.841126,18.882518,3.591155,8.183754,10.968536
воронежская область,100.98,100.14,7.888888,4.126813,7.263614,7.086193,18.362442,6.209715,3.916095,1.615044,5.277875,3.107542,7.296134,17.212672,3.783786,7.262683,11.960398
ивановская область,100.58,100.49,12.176576,6.392741,7.11949,7.354808,19.704542,6.570194,3.447797,3.034248,6.066,2.86845,6.635719,19.531094,3.071259,7.81917,11.53296
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
амурская область,100.71,100.7,9.400701,7.635369,7.177096,7.675617,15.827116,7.918382,2.936938,1.945715,5.353542,4.571315,6.311061,12.589005,6.35249,8.165421,10.615315
магаданская область,101.47,100.37,8.539257,9.210575,8.640861,9.013332,12.483261,9.005738,2.751284,1.860154,5.526068,2.420272,5.705916,12.666801,6.34615,6.48213,8.107269
сахалинская область,100.29,100.61,10.015007,6.363711,5.941075,6.520941,12.917196,5.974197,3.935203,1.244429,3.709194,3.554248,4.534015,13.765076,4.463828,8.412891,8.527331
еврейская автономная область,101.68,100.26,9.50863,8.875623,6.449767,8.510016,16.620976,7.180597,5.185748,2.618358,5.287589,4.642706,6.084733,17.240324,4.361181,8.302315,9.534358


In [ ]:
temp_df = pd.read_excel('правильное наименование.xlsx')
dict_key = {n:k for k, n in zip(temp_df['key'], temp_df['name'])}


In [45]:
infl = pd.read_excel('infl_fed.xls')
list_months = infl.iloc[2, 2:].to_list()
dict_months_to_num = {k:f'0{i}' if i // 10 == 0 else f'{i}' for k, i in zip(infl.iloc[2, 2:].unique().tolist(), range(1,13))}
list_col = ['region', 'type_of_infl']
for year in range(2010, 2026):
    temp_list = list_months[12*(year-2010):12*(year-2009)]
    list_col += [dict_months_to_num[i] + f"_{year}" for i in temp_list]
infl_df = infl.iloc[4:, :].copy(deep = True).reset_index(drop=True).set_axis(list_col, axis=1)

In [47]:
[i in list(dict_key.keys()) for i in infl_df['region']]

[False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,


In [48]:
list(dict_key.keys())

['Белгородская область',
 'Брянская область',
 'Владимирская область',
 'Воронежская область',
 'Ивановская область',
 'Калужская область',
 'Костромская область',
 'Курская область',
 'Липецкая область',
 'Московская область',
 'Орловская область',
 'Рязанская область',
 'Смоленская область',
 'Тамбовская область',
 'Тверская область',
 'Тульская область',
 'Ярославская область',
 'Город Москва столица Российской Федерации город федерального значения',
 'Республика Карелия',
 'Республика Коми',
 'Ненецкий автономный округ (Архангельская область)',
 'Архангельская область (кроме Ненецкого автономного округа)',
 'Вологодская область',
 'Калининградская область',
 'Ленинградская область',
 'Мурманская область',
 'Новгородская область',
 'Псковская область',
 'Город Санкт-Петербург город федерального значения',
 'Республика Адыгея (Адыгея)',
 'Республика Калмыкия',
 'Республика Крым',
 'Краснодарский край',
 'Астраханская область',
 'Волгоградская область',
 'Ростовская область',
 'Город 

In [12]:
infl_df = infl_df[(infl_df['type_of_infl'] == 'Все товары и услуги')].copy(deep = True).drop('type_of_infl')

KeyError: "['type_of_infl'] not found in axis"

In [4]:
infl_df

84

In [28]:
list_regions_infl

['Белгородская область',
 'Брянская область',
 'Владимирская область',
 'Воронежская область',
 'Ивановская область',
 'Калужская область',
 'Костромская область',
 'Курская область',
 'Липецкая область',
 'Московская область',
 'Орловская область',
 'Рязанская область',
 'Смоленская область',
 'Тамбовская область',
 'Тверская область',
 'Тульская область',
 'Ярославская область',
 'Город Москва столица Российской Федерации город федерального значения',
 'Республика Карелия',
 'Республика Коми',
 'Архангельская область',
 'Ненецкий автономный округ (Архангельская область)',
 'Архангельская область (кроме Ненецкого автономного округа)',
 'Вологодская область',
 'Калининградская область',
 'Ленинградская область',
 'Мурманская область',
 'Новгородская область',
 'Псковская область',
 'Город Санкт-Петербург город федерального значения',
 'Республика Адыгея (Адыгея)',
 'Республика Калмыкия',
 'Республика Крым',
 'Краснодарский край',
 'Астраханская область',
 'Волгоградская область',
 'Рос

In [5]:
infl_df

,region,type_of_infl,01_2010,02_2010,03_2010,04_2010,05_2010,06_2010,07_2010,08_2010,...,02_2024,03_2024,04_2025,05_2025,06_2025,07_2025,08_2025,09_2025,10_2025,11_2025
0,Белгородская область,Все товары и услуги,102.08,100.76,100.63,100.14,100.18,100.28,100.43,100.89,...,100.82,100.54,100.62,100.38,100.04,100.53,99.79,100.41,100.42,100.63
1,Белгородская область,Все товары,100.63,100.56,100.74,99.99,100.21,100.35,100.53,101.19,...,100.86,100.46,100.55,100.17,99.86,99.83,100.02,100.31,100.88,100.6
2,Белгородская область,Продовольственные товары,101.16,101.15,101.24,99.85,100.09,100.46,100.7,101.67,...,101.12,100.48,101.06,100.46,99.9,99.41,99.56,100.13,101.11,101.02
3,Белгородская область,Непродовольственные товары,99.96,99.8,100.1,100.18,100.36,100.2,100.31,100.56,...,100.6,100.45,100.02,99.87,99.81,100.26,100.49,100.5,100.65,100.18
4,Брянская область,Все товары и услуги,102.37,100.9,100.62,100.34,100.31,100.44,100.49,100.77,...,101.06,100.53,100.52,100.76,100.09,100.55,99.84,99.85,100.23,100.33
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
331,Еврейская автономная область,Непродовольственные товары,100.94,100.62,100.71,100.24,100.35,100.58,100.34,100.29,...,100.57,100.32,100.34,100.25,99.91,100.43,100.3,100.14,102.3,100.22
332,Чукотский автономный округ,Все товары и услуги,100.46,100.9,101.24,101.08,100.69,100.17,97.88,98.35,...,100.95,100.38,99.62,99.64,99.82,100.27,100.26,100.01,100.24,100.41
333,Чукотский автономный округ,Все товары,100.6,101.1,101.31,101.31,100.77,100.21,97.33,97.9,...,100.58,100.59,99.84,99.91,99.92,99.35,100.35,100,100.31,100.46
334,Чукотский автономный округ,Продовольственные товары,100.78,101.36,101.55,101.7,100.95,100.21,96.39,97.12,...,100.45,100.8,100.25,100.11,100.04,99.7,99.87,100.56,100.32,100.49


In [32]:
df = df[~(df.index.get_level_values('region') == 'ханты-мансийский автономный округ')].copy(deep=True)

Доля дотаций в МБТ  Дефицит отношение  \
region                     year                                          
белгородская область       2019            0.069876           0.934485   
                           2020            0.109277           0.932991   
                           2021            0.099364           1.109833   
                           2022            0.068490           0.804803   
                           2023            0.335542           0.891685   
...                                             ...                ...   
чукотский автономный округ 2019            0.334060           0.986916   
                           2020            0.577669           0.987378   
                           2021            0.556139           0.890232   
                           2022            0.544729           0.874682   
                           2023            0.533627           0.966426   

                                 Рост ВРП  Доля собственных доходов  \
region                     year                                       
белгородская область       2019  1.054232                  0.627457   
                           2020  1.044935                  0.568333   
                           2021  1.030415                  0.673187   
                           2022  1.124929                  0.604290   
                           2023  1.087354                  0.590858   
...                                   ...                       ...   
чукотский автономный округ 2019  1.049464                  0.215443   
                           2020  1.092387                  0.387937   
                           2021  1.128682                  0.329120   
                           2022  1.138072                  0.300246   
                           2023  1.065409                  0.342546   

                                 Социальные расходы  Дефицит исполненный  \
region                     year                                            
белгородская область       2019            0.628825            -0.090048   
                           2020            0.686004             0.016836   
                           2021            0.689167             2.422311   
                           2022            0.655199            -2.274987   
                           2023            0.649981            -0.271843   
...                                             ...                  ...   
чукотский автономный округ 2019            0.253758             0.396202   
                           2020            0.334768             3.711364   
                           2021            0.330224            -1.405542   
                           2022            0.362133            -3.462239   
                           2023            0.366073            -0.637730   

                                 Дефицит планируемый  Константа     ДоИПР  \
region                     year                                             
белгородская область       2019            -1.010103          1  0.148893   
                           2020            -1.069160          1  0.117556   
                           2021             1.482396          1  0.094571   
                           2022            -3.492084          1  0.333937   
                           2023            -1.950334          1  0.348040   
...                                              ...        ...       ...   
чукотский автономный округ 2019            -0.814737          1  2.525028   
                           2020            -0.573064          1  2.681575   
                           2021            -4.823418          1  2.221114   
                           2022            -5.501735          1  1.774252   
                           2023            -1.177462          1  1.908655   

                                 Изменение долговых расходов  ...  \
region                     year                               ...   
белгородская обла

In [33]:
df

Доля дотаций в МБТ  Дефицит отношение  \
region                     year                                          
белгородская область       2019            0.069876           0.934485   
                           2020            0.109277           0.932991   
                           2021            0.099364           1.109833   
                           2022            0.068490           0.804803   
                           2023            0.335542           0.891685   
...                                             ...                ...   
чукотский автономный округ 2019            0.334060           0.986916   
                           2020            0.577669           0.987378   
                           2021            0.556139           0.890232   
                           2022            0.544729           0.874682   
                           2023            0.533627           0.966426   

                                 Рост ВРП  Доля собственных доходов  \
region                     year                                       
белгородская область       2019  1.054232                  0.627457   
                           2020  1.044935                  0.568333   
                           2021  1.030415                  0.673187   
                           2022  1.124929                  0.604290   
                           2023  1.087354                  0.590858   
...                                   ...                       ...   
чукотский автономный округ 2019  1.049464                  0.215443   
                           2020  1.092387                  0.387937   
                           2021  1.128682                  0.329120   
                           2022  1.138072                  0.300246   
                           2023  1.065409                  0.342546   

                                 Социальные расходы  Дефицит исполненный  \
region                     year                                            
белгородская область       2019            0.628825            -0.090048   
                           2020            0.686004             0.016836   
                           2021            0.689167             2.422311   
                           2022            0.655199            -2.274987   
                           2023            0.649981            -0.271843   
...                                             ...                  ...   
чукотский автономный округ 2019            0.253758             0.396202   
                           2020            0.334768             3.711364   
                           2021            0.330224            -1.405542   
                           2022            0.362133            -3.462239   
                           2023            0.366073            -0.637730   

                                 Дефицит планируемый  Константа     ДоИПР  \
region                     year                                             
белгородская область       2019            -1.010103          1  0.148893   
                           2020            -1.069160          1  0.117556   
                           2021             1.482396          1  0.094571   
                           2022            -3.492084          1  0.333937   
                           2023            -1.950334          1  0.348040   
...                                              ...        ...       ...   
чукотский автономный округ 2019            -0.814737          1  2.525028   
                           2020            -0.573064          1  2.681575   
                           2021            -4.823418          1  2.221114   
                           2022            -5.501735          1  1.774252   
                           2023            -1.177462          1  1.908655   

                                 Изменение долговых расходов  ...  \
region                     year                               ...   
белгородская обла

In [13]:
np.unique([i.lower()[8:] for i in infl_df['region'].to_list()])

array(['    архангельская область (кроме ненецкого автономного округа)',
       '    ненецкий автономный округ (архангельская область)',
       'алтайский край', 'амурская область', 'архангельская область',
       'астраханская область', 'белгородская область', 'брянская область',
       'владимирская область', 'волгоградская область',
       'вологодская область', 'воронежская область',
       'город москва столица российской федерации город федерального значения',
       'город санкт-петербург город федерального значения',
       'город федерального значения севастополь',
       'еврейская автономная область', 'забайкальский край',
       'ивановская область', 'иркутская область',
       'кабардино-балкарская республика', 'калининградская область',
       'калужская область', 'камчатский край',
       'карачаево-черкесская республика', 'кемеровская область - кузбасс',
       'кировская область', 'костромская область', 'краснодарский край',
       'красноярский край', 'курганская обла

In [11]:
df.index

MultiIndex([(        'белгородская область', 2019),
            (        'белгородская область', 2020),
            (        'белгородская область', 2021),
            (        'белгородская область', 2022),
            (        'белгородская область', 2023),
            (            'брянская область', 2019),
            (            'брянская область', 2020),
            (            'брянская область', 2021),
            (            'брянская область', 2022),
            (            'брянская область', 2023),
            ...
            ('еврейская автономная область', 2019),
            ('еврейская автономная область', 2020),
            ('еврейская автономная область', 2021),
            ('еврейская автономная область', 2022),
            ('еврейская автономная область', 2023),
            (  'чукотский автономный округ', 2019),
            (  'чукотский автономный округ', 2020),
            (  'чукотский автономный округ', 2021),
            (  'чукотский автономный округ', 202

In [11]:
list_col = []
for year in range(2010, 2026):
    temp_list = list_months[12*(year-2010):12*(year-2009)]
    list_col += [dict_months_to_num[i] + f"_{year}" for i in temp_list]


In [12]:
list_col

['01_2010',
 '02_2010',
 '03_2010',
 '04_2010',
 '05_2010',
 '06_2010',
 '07_2010',
 '08_2010',
 '09_2010',
 '10_2010',
 '11_2010',
 '12_2010',
 '01_2011',
 '02_2011',
 '03_2011',
 '04_2011',
 '05_2011',
 '06_2011',
 '07_2011',
 '08_2011',
 '09_2011',
 '10_2011',
 '11_2011',
 '12_2011',
 '01_2012',
 '02_2012',
 '03_2012',
 '04_2012',
 '05_2012',
 '06_2012',
 '07_2012',
 '08_2012',
 '09_2012',
 '10_2012',
 '11_2012',
 '12_2012',
 '01_2013',
 '02_2013',
 '03_2013',
 '04_2013',
 '05_2013',
 '06_2013',
 '07_2013',
 '08_2013',
 '09_2013',
 '10_2013',
 '11_2013',
 '12_2013',
 '01_2014',
 '02_2014',
 '05_2014',
 '06_2014',
 '08_2014',
 '09_2014',
 '10_2014',
 '11_2014',
 '12_2014',
 '01_2014',
 '02_2014',
 '03_2014',
 '04_2015',
 '05_2015',
 '06_2015',
 '07_2015',
 '08_2015',
 '09_2015',
 '10_2015',
 '11_2015',
 '12_2015',
 '01_2015',
 '02_2015',
 '03_2015',
 '04_2016',
 '05_2016',
 '06_2016',
 '07_2016',
 '08_2016',
 '09_2016',
 '10_2016',
 '11_2016',
 '12_2016',
 '01_2016',
 '02_2016',
 '03